# PigeonPilot — Interactive Playground

Load a **finished** named run from `Models.ipynb` (Step 10), review its test metrics, then draw a displacement route and watch both pigeons fly it and read out a home direction.

**Contract:** the readout predicts a **home heading bin** (10° resolution), not a return polyline. The homebound leg is a straight ray at the predicted heading, drawn at the *true* home distance — it demonstrates a direction, it does not demonstrate navigation.

Every number shown by the widget is computed in the kernel from the loaded checkpoint. The browser only replays a precomputed flight plan; it never estimates anything.

Requires: kernel **Python 3.11 (SNN Hackathon)**, `anywidget` (`pip install anywidget`), and at least one run under `outputs/checkpoints/`. The old Matplotlib playground remains available via `pigeonpilot.playground.launch_playground`.


### 1. Imports + pick a run


In [ ]:
from IPython.display import Markdown, display

from pigeonpilot.snn import list_runs, load_run

runs = list_runs()
assert runs, "No checkpoints yet — run Models.ipynb Step 10 (save_run) first."

display(Markdown("### Available runs"))
for r in runs:
    mark = " ← latest" if r["is_latest"] else ""
    display(Markdown(f"- `{r['name']}` · n_res={r['n_reservoir']}{mark}"))

# Change this to a concrete name, e.g. "n10000_main", or keep "latest"
RUN_NAME = "latest"
print("will load:", RUN_NAME)


### Available runs

- `n10_main` · n_res=10 ← latest

- `n50_main` · n_res=50

- `n10000_main` · n_res=10000

- `n1000_main` · n_res=1000

- `n100_main` · n_res=100

will load: latest


### 2. Load checkpoint + show jury metrics


In [ ]:
bundle = load_run(RUN_NAME)
cfg = bundle.config
metrics = bundle.metrics

display(Markdown(
    f"### Loaded `{RUN_NAME}`\n"
    f"- reservoir size: **{cfg.n_reservoir}**\n"
    f"- encoding: v=`{cfg.encoding_velocity:.4f}`, dt=`{cfg.encoding_dt}`, "
    f"rate=`{cfg.input_rate_hz}` Hz, silence=`{cfg.trailing_silence}`\n"
    f"- ridge α=`{cfg.ridge_alpha}`"
))

summary = metrics.get("summary") or {}
if summary:
    lines = ["### Test angular error (from training run)\n"]
    for name in ("A", "B"):
        if name not in summary:
            continue
        s = summary[name]
        lines.append(
            f"- **Pigeon {name}**: mean error {s['mean_deg']:.1f}° ± {s['std_deg']:.1f}° "
            f"| exact-bin acc {100 * s['exact_acc']:.1f}%"
        )
    by_diff = metrics.get("by_difficulty") or {}
    if by_diff:
        lines.append("\n| difficulty | A mean ° | B mean ° | n |")
        lines.append("|---|---:|---:|---:|")
        for diff, row in by_diff.items():
            lines.append(
                f"| {diff} | {row['A_mean_deg']:.1f} | {row['B_mean_deg']:.1f} | {row['n']} |"
            )
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("_No metrics stored in this checkpoint — demo still works._"))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### Loaded `latest`
- reservoir size: **10**
- encoding: v=`0.0091`, dt=`1.0`, rate=`40.0` Hz, silence=`20`
- ridge α=`0.01`

### Test angular error (from training run)

- **Pigeon A**: mean error 78.2° ± 51.9° | exact-bin acc 6.8%
- **Pigeon B**: mean error 68.4° ± 56.4° | exact-bin acc 12.9%

| difficulty | A mean ° | B mean ° | n |
|---|---:|---:|---:|
| easy | 74.4 | 53.3 | 36 |
| medium | 74.6 | 76.4 | 28 |
| hard | 78.5 | 65.0 | 26 |
| expert | 77.6 | 63.1 | 29 |
| extreme | 86.8 | 88.2 | 28 |

### 3. Draw → displace → predict

1. **Draw** a route with the mouse, starting at the green home star.
2. Click **Fly**. The bird flies your route while the spike raster below fills in *live* — each tick is the body-ring neuron that points North on that leg.
3. At the release point both pigeons read out a home direction and fly it **side by side**: A (fixed reservoir) and B (STDP). The blue dashed line is the true home vector.
4. The rings show each model's full 36-bin direction profile.

**What the demo does to your stroke, and why.** Pointer samples carry hand tremor, and every sample step used to be snapped separately — a stroke you see as ~6 units long arrived at the encoder as 15–53 units with 100+ heading changes. The curriculum only contains routes of 1.08–9.15 total length with at most 8 heading blocks, so that input was far outside anything the reservoir was trained on. The stroke is therefore simplified (Ramer–Douglas–Peucker) and uniformly scaled into the training band. **Shape is preserved, only length changes**, and the applied factor is printed under the canvas.

**Why one bird out and two birds back.** A and B are not two birds in the dataset — they are two reservoirs (fixed vs. plastic) receiving the *same* spike train. On the outbound leg they are indistinguishable by construction, so showing two would be theatre. They differ only in what they read out at the release point, which is exactly where the second bird appears.

**Reading the numbers.** The readout returns a *direction* (1 of 36 bins), not a return path, so the homebound leg borrows the true distance. A uniform random guess scores 90° mean circular error; this checkpoint's test-set means are in the table above and are repeated on each model card.

**One route is not a result — and the widget says so.** The per-route spread is larger than the gap between the two models, so individual flights routinely disagree with the average: on the held-out split the two models tie on roughly half the routes and each wins a share of the rest. Seeing B beat A on a drawn path is therefore expected, not a contradiction of `Models_staged.ipynb`. Every card carries the model's full error histogram with the current flight marked on it, plus the head-to-head win rates measured on the held-out routes — all read from this checkpoint, so none of it can drift out of date.

> **If the widget sits on "Running the reservoir …"** the kernel is holding an older `pigeonpilot` in memory — Python does not re-import edited modules into a running kernel. Restart the kernel and run all cells. Run `playground.self_test()` to confirm the kernel side is healthy.


In [ ]:
from pigeonpilot.widget import PigeonPlayground

# One canvas, both pigeons; set models=("A",) to run a single pigeon.
#
# reference=True scores the 147 held-out routes once (~8 s, then cached next to
# the checkpoint) so every flight can be shown against the error distribution
# it was drawn from.
playground = PigeonPlayground(bundle, models=("A", "B"), reference=True)
playground
